### Install Required Packages

In [3]:
!pip install sentence-transformers faiss-cpu numpy

In [5]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


### Import Libraries

In [6]:
import pickle
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

### Load chunks.pkl

In [7]:
with open("data/chunks.pkl", "rb") as f:

    chunks = pickle.load(f)

print("Chunks Loaded:", len(chunks))

Chunks Loaded: 6


### Preview Sample Chunks

In [9]:
for chunk in chunks[:3]:

    print("Source:", chunk["source"])
    print(chunk["text"])
    print("-" * 80)

Source: cloud_migration_guide.txt
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.
--------------------------------------------------------------------------------
Source: company_policies.txt
Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.
--------------------------------------------------------------------------------
Source: employee_handbook.txt
Employee Handbook

Employees receive 20 annual leave days.

Remote work is allowed three days per week.

Working hours are 9 AM to 6 PM.

Health insurance benefits are provided.
--------------------------------------------------------------------------------


### Extract Text from Chunks

In [10]:
texts = []

for chunk in chunks:

    texts.append(chunk["text"])

print("Total Text Chunks:", len(texts))

Total Text Chunks: 6


### Initialize TF-IDF Vectorizer

In [11]:
vectorizer = TfidfVectorizer()

### Generate Embeddings

In [12]:
embeddings = vectorizer.fit_transform(texts)

### Convert Sparse Matrix to NumPy Array

In [13]:
embeddings = embeddings.toarray().astype("float32")

print("Embedding Shape:", embeddings.shape)

Embedding Shape: (6, 112)


### Create FAISS Index

In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

Vectors Stored: 6


### Save Embeddings

In [15]:
np.save(
    "data/embeddings.npy",
    embeddings
)

print("embeddings.npy saved successfully!")

embeddings.npy saved successfully!


### Save Metadata

In [16]:
metadata = []

for chunk in chunks:

    metadata.append({
        "source": chunk["source"],
        "text": chunk["text"]
    })

with open("data/metadata.pkl", "wb") as f:

    pickle.dump(metadata, f)

print("metadata.pkl saved successfully!")

metadata.pkl saved successfully!


### Save Vectorizer

We must save the TF-IDF vocabulary so we can embed future queries.

In [17]:
with open("data/vectorizer.pkl", "wb") as f:

    pickle.dump(vectorizer, f)

print("vectorizer.pkl saved successfully!")

vectorizer.pkl saved successfully!


### Save FAISS Index

In [18]:
faiss.write_index(
    index,
    "data/faiss_index.bin"
)

print("faiss_index.bin saved successfully!")

faiss_index.bin saved successfully!


### Verify Files

In [19]:
import os

print(os.path.exists("data/embeddings.npy"))
print(os.path.exists("data/metadata.pkl"))
print(os.path.exists("data/vectorizer.pkl"))
print(os.path.exists("data/faiss_index.bin"))

True
True
True
True


### Test Query

In [20]:
query = "What is the password policy?"

### Convert Query to Vector

In [21]:
query_vector = vectorizer.transform([query])

query_vector = query_vector.toarray().astype("float32")

### Search FAISS

In [22]:
k = 3

distances, indices = index.search(
    query_vector,
    k
)

### Display Results

In [23]:
for idx in indices[0]:

    print("Source:", metadata[idx]["source"])
    print(metadata[idx]["text"])
    print("-" * 80)

Source: cloud_migration_guide.txt
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.
--------------------------------------------------------------------------------
Source: support_tickets.csv
TicketID           Issue Priority
0     T001   Login failure     High
1     T002       VPN issue   Medium
2     T003  Password reset      Low
--------------------------------------------------------------------------------
Source: company_policies.txt
Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.
--------------------------------------------------------------------------------
